# MindForge Initiative
DeepSeek Chat History → Obsidian Exporter

Exports all DeepSeek web conversations to Markdown files with YAML frontmatter, ready for Obsidian import.

**Approach**: Playwright handles login & cookie capture only. Data fetching uses DeepSeek's internal API for speed and reliability.

### Install Dependencies & Import Libraries
Run this cell once to install required packages. Subsequent runs will skip installation if packages are already present.

In [8]:
# ── Install dependencies (skip if already installed) ──
import subprocess, sys

def install_if_missing(package, import_name=None):
    """Install a package only if it's not already importable."""
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

install_if_missing("playwright")
install_if_missing("markdownify")
install_if_missing("requests")
install_if_missing("python-dotenv", "dotenv")

# ── Install Playwright browser binaries (Chromium only) ──
print("\nInstalling Playwright Chromium browser...")
result = subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Playwright Chromium browser ready")
else:
    print(f"⚠ Playwright browser install issue:\n{result.stderr}")


✓ playwright already installed
✓ markdownify already installed
✓ requests already installed
Installing python-dotenv...
✓ python-dotenv installed

Installing Playwright Chromium browser...
✓ Playwright Chromium browser ready


In [9]:
# ── Import all required libraries ──
import json
import os
import re
import time
import logging
from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import urljoin

import requests
import markdownify
from dotenv import load_dotenv

print("\n── All imports successful ──")
print(f"Python:      {sys.version.split()[0]}")
print(f"Requests:    {requests.__version__}")
print(f"Markdownify: installed ✓")


── All imports successful ──
Python:      3.12.10
Requests:    2.33.1
Markdownify: installed ✓


### Configuration Constants
Paths, URLs, retry settings, and other constants used throughout the notebook. Adjust as needed.

In [ ]:
# ── Paths ──
BASE_DIR = Path("./")
EXPORT_DIR = BASE_DIR / "DeepSeek_Exports"
COOKIE_FILE = BASE_DIR / "deepseek_cookies.json"
MANIFEST_FILE = EXPORT_DIR / "conversations_manifest.json"
FAILURE_LOG = EXPORT_DIR / "failed_exports.log"

# ── Obsidian Vault (loaded from .env) ──
load_dotenv(override=True)
_vault_path = os.environ.get("OBSIDIAN_VAULT_PATH", "").strip()
if not _vault_path or "path\\to\\your" in _vault_path:
    raise ValueError(
        "OBSIDIAN_VAULT_PATH not set!\n"
        "1. Open Obsidian → Settings (gear icon) → About → copy the Vault path\n"
        "2. Paste it in .env:  OBSIDIAN_VAULT_PATH=C:\\Users\\...\\MyVault"
    )
OBSIDIAN_VAULT_DIR = Path(_vault_path)
OBSIDIAN_SUBFOLDER = "DeepSeek"  # Files go into vault/DeepSeek/

# ── DeepSeek URLs ──
DEEPSEEK_BASE_URL = "https://chat.deepseek.com"
DEEPSEEK_LOGIN_URL = "https://chat.deepseek.com/sign_in"
DEEPSEEK_API_BASE = "https://chat.deepseek.com/api/v0"

# ── Retry & Timing ──
MAX_RETRIES = 3
RETRY_DELAY_SEC = 2
REQUEST_DELAY_SEC = 1.0   # Politeness delay between API calls
LOGIN_TIMEOUT_SEC = 120   # How long to wait for manual Google login
PAGE_LOAD_TIMEOUT_MS = 30_000

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("deepseek_export")

# ── Create export directory ──
EXPORT_DIR.mkdir(exist_ok=True)

print(f"Export dir:   {EXPORT_DIR.resolve()}")
print(f"Cookie file:  {COOKIE_FILE.resolve()}")
print(f"Manifest:     {MANIFEST_FILE.resolve()}")
print(f"Base URL:     {DEEPSEEK_BASE_URL}")
print(f"API base:     {DEEPSEEK_API_BASE}")
print(f"Obsidian:     {OBSIDIAN_VAULT_DIR / OBSIDIAN_SUBFOLDER}")

Export dir:   C:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\DeepSeek_Exports
Cookie file:  C:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\deepseek_cookies.json
Manifest:     C:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\DeepSeek_Exports\conversations_manifest.json
Base URL:     https://chat.deepseek.com
API base:     https://chat.deepseek.com/api/v0


### Load Auth Token
Reads the `DEEPSEEK_TOKEN` from `.env` file and builds an authenticated HTTP session.

**How to get your token**: Open DeepSeek in Chrome → F12 → Network tab → click any chat → find an API request → copy the `Authorization: Bearer xxx` value → paste just the token part into `.env`.

In [11]:
# ── Load token from .env ──
load_dotenv()

auth_token = os.environ.get("DEEPSEEK_TOKEN", "").strip()
if not auth_token or auth_token == "paste_your_token_here":
    raise ValueError(
        "DEEPSEEK_TOKEN not set!\n"
        "Option A (easiest):\n"
        "  1. Open https://chat.deepseek.com in Chrome (logged in)\n"
        "  2. F12 → Console tab → run:\n"
        '     JSON.parse(localStorage.getItem("userToken")).value\n'
        "  3. Copy the token string\n"
        "Option B (Network tab):\n"
        "  1. F12 → Network tab → click any chat\n"
        "  2. Find an API request → Headers → copy the Bearer token\n"
        "Paste it in .env:  DEEPSEEK_TOKEN=eyJhbG..."
    )

# ── Build authenticated HTTP session ──
http_session = requests.Session()
http_session.headers.update({
    "Authorization": f"Bearer {auth_token}",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
    "x-client-platform": "web",
    "x-client-locale": "en_US",
})

# ── Quick validation: test the token ──
try:
    test_resp = http_session.get(f"{DEEPSEEK_API_BASE}/chat_session/fetch_page", timeout=10)
    if test_resp.status_code == 200:
        body = test_resp.json()
        sessions = body.get("data", {}).get("biz_data", {}).get("chat_sessions", [])
        print(f"✓ Token is valid — found {len(sessions)} conversations on first page")
    elif test_resp.status_code == 401:
        print("✗ Token is expired or invalid — get a fresh one from Chrome DevTools")
    else:
        print(f"⚠ Unexpected status {test_resp.status_code} — token may still work")
except requests.ConnectionError:
    print("⚠ Cannot reach DeepSeek — check your network (token saved for later)")

print(f"  Token:   {auth_token[:20]}...{auth_token[-10:]}")
print(f"  Session: ready")

ValueError: DEEPSEEK_TOKEN not set!
Option A (easiest):
  1. Open https://chat.deepseek.com in Chrome (logged in)
  2. F12 → Console tab → run:
     JSON.parse(localStorage.getItem("userToken")).value
  3. Copy the token string
Option B (Network tab):
  1. F12 → Network tab → click any chat
  2. Find an API request → Headers → copy the Bearer token
Paste it in .env:  DEEPSEEK_TOKEN=eyJhbG...

### Fetch All Conversations
Uses DeepSeek's internal API to fetch the conversation list. Falls back to DOM scraping via Playwright subprocess if the API doesn't work.

In [ ]:
def api_request(endpoint, method="GET", **kwargs):
    """Make an API request with retry logic."""
    url = f"{DEEPSEEK_API_BASE}/{endpoint.lstrip('/')}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = http_session.request(method, url, timeout=30, **kwargs)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 401:
                log.error("Auth token expired — re-run the login cell")
                raise PermissionError("Auth token expired")
            else:
                log.warning(f"API {endpoint} returned {resp.status_code} (attempt {attempt}/{MAX_RETRIES})")
        except (requests.ConnectionError, requests.Timeout) as e:
            log.warning(f"API {endpoint} network error (attempt {attempt}/{MAX_RETRIES}): {e}")
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_DELAY_SEC)
    raise RuntimeError(f"API {endpoint} failed after {MAX_RETRIES} attempts")


def fetch_conversations_api():
    """Fetch ALL conversations via DeepSeek's internal API with pagination."""
    log.info("Fetching conversations via API...")
    all_conversations = []
    last_seq_id = None
    page = 0

    while True:
        page += 1
        endpoint = "chat_session/fetch_page"
        if last_seq_id is not None:
            endpoint += f"?before_seq_id={last_seq_id}"

        data = api_request(endpoint)
        biz_data = data.get("data", {}).get("biz_data", {})
        sessions = biz_data.get("chat_sessions", [])
        has_more = biz_data.get("has_more", False)

        if not sessions:
            break

        for item in sessions:
            conv_id = str(item.get("id", ""))
            title = item.get("title") or "Untitled"
            updated_at = item.get("updated_at") or ""
            created_at = item.get("created_at") or ""

            # Parse date — could be ISO string or timestamp
            raw_date = updated_at or created_at
            if raw_date and isinstance(raw_date, (int, float)):
                date = datetime.fromtimestamp(raw_date).strftime("%Y-%m-%d")
            elif raw_date and isinstance(raw_date, str):
                try:
                    date = datetime.fromisoformat(raw_date.replace("Z", "+00:00")).strftime("%Y-%m-%d")
                except ValueError:
                    date = raw_date[:10] if len(raw_date) >= 10 else "unknown-date"
            else:
                date = "unknown-date"

            all_conversations.append({
                "conversation_id": conv_id,
                "original_title": title,
                "date": date,
                "url": f"{DEEPSEEK_BASE_URL}/a/{conv_id}",
            })

            # Track seq_id for pagination
            seq_id = item.get("seq_id")
            if seq_id is not None:
                if last_seq_id is None or seq_id < last_seq_id:
                    last_seq_id = seq_id

        log.info(f"  Page {page}: got {len(sessions)} sessions (total so far: {len(all_conversations)})")

        if not has_more:
            break
        time.sleep(REQUEST_DELAY_SEC)

    log.info(f"Fetched {len(all_conversations)} conversations total across {page} page(s)")
    return all_conversations if all_conversations else None


def fetch_conversations_fallback():
    """Fallback: use scripts/pw_fetch_convos.py to scroll sidebar and scrape conversation list."""
    log.info("API didn't return conversations. Falling back to DOM scraping...")

    script_path = SCRIPTS_DIR / "pw_fetch_convos.py"
    result = subprocess.run(
        [sys.executable, str(script_path), str(COOKIE_FILE), str(PAGE_LOAD_TIMEOUT_MS)],
        capture_output=True, text=True, timeout=120
    )

    if result.returncode != 0:
        raise RuntimeError(f"Fallback scraping failed:\n{result.stderr[-500:]}")

    for line in result.stdout.splitlines():
        if line.startswith("RESULT:"):
            convos = json.loads(line[7:])
            for c in convos:
                c.setdefault("date", "unknown-date")
            return convos

    raise RuntimeError("No result from fallback scraping")


def fetch_all_conversations():
    """Fetch all conversations — API first, DOM fallback second."""
    try:
        convos = fetch_conversations_api()
        if convos:
            return convos
    except PermissionError:
        raise
    except Exception as e:
        log.warning(f"API approach failed: {e}")

    return fetch_conversations_fallback()


# ── Fetch conversations ──
conversations = fetch_all_conversations()
print(f"\n✓ Found {len(conversations)} conversations")
for c in conversations[:5]:
    print(f"  [{c['date']}] {c['original_title'][:50]}  (id: {c['conversation_id'][:8]}...)")
if len(conversations) > 5:
    print(f"  ... and {len(conversations) - 5} more")

### Parse Single Conversation Messages
Fetches all messages for a given conversation via API (with DOM fallback). Skips thinking/reasoning tokens and image attachments. Preserves Markdown formatting.

In [ ]:
def fetch_messages_api(conversation_id):
    """Fetch messages for a conversation via the API."""
    endpoint = f"chat/history_messages?chat_session_id={conversation_id}"
    try:
        data = api_request(endpoint)
        if not data:
            return None

        messages_raw = data.get("data", {}).get("biz_data", {}).get("chat_messages", [])

        if not messages_raw:
            return None

        messages = []
        for msg in messages_raw:
            role = msg.get("role", "").lower()
            if role not in ("user", "assistant"):
                continue
            content = msg.get("content", "")
            if not content.strip():
                continue
            if role == "assistant":
                content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
                content = re.sub(r"<reasoning>.*?</reasoning>", "", content, flags=re.DOTALL).strip()
            if content.strip():
                messages.append({"role": role, "content": content.strip()})

        log.info(f"  Fetched {len(messages)} messages via API for {conversation_id[:8]}...")
        return messages
    except (RuntimeError, PermissionError):
        raise
    except Exception as e:
        log.debug(f"  API endpoint failed for {conversation_id[:8]}: {e}")
        return None


def fetch_messages_fallback(conversation_id, conv_url):
    """Fallback: scrape messages using scripts/pw_fetch_messages.py."""
    log.info(f"  Falling back to DOM scraping for {conversation_id[:8]}...")

    script_path = SCRIPTS_DIR / "pw_fetch_messages.py"
    result = subprocess.run(
        [sys.executable, str(script_path), str(COOKIE_FILE), conv_url, str(PAGE_LOAD_TIMEOUT_MS)],
        capture_output=True, text=True, timeout=60
    )

    if result.returncode != 0:
        raise RuntimeError(f"DOM scraping failed for {conversation_id[:8]}: {result.stderr[-300:]}")

    for line in result.stdout.splitlines():
        if line.startswith("RESULT:"):
            raw_msgs = json.loads(line[7:])
            messages = []
            for msg in raw_msgs:
                html = msg.get("html", "")
                if html:
                    content = markdownify.markdownify(html, heading_style="ATX", strip=["img"])
                else:
                    content = msg.get("content", "")
                content = content.strip()
                if content:
                    messages.append({"role": msg["role"], "content": content})
            return messages

    raise RuntimeError(f"No result from DOM scraping for {conversation_id[:8]}")


def fetch_messages(conversation_id, conv_url):
    """Fetch messages for a conversation — API first, DOM fallback."""
    try:
        msgs = fetch_messages_api(conversation_id)
        if msgs:
            return msgs
    except PermissionError:
        raise
    except Exception as e:
        log.debug(f"  API failed for {conversation_id[:8]}: {e}")

    return fetch_messages_fallback(conversation_id, conv_url)


# ── Quick test (will only work when logged in and on open network) ──
print("✓ Message parsing functions defined")
print("  fetch_messages(conversation_id, url) → list of {role, content}")

### Generate Markdown Files & Update Manifest
Converts each conversation into an Obsidian-ready `.md` file with YAML frontmatter. Maintains `conversations_manifest.json` for incremental updates.

In [ ]:
def load_manifest():
    """Load existing manifest or return empty structure."""
    if MANIFEST_FILE.exists():
        with open(MANIFEST_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"last_updated": None, "conversations": []}


def save_manifest(manifest):
    """Save manifest to JSON file."""
    manifest["last_updated"] = datetime.now().isoformat()
    with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    log.info(f"Manifest updated ({len(manifest['conversations'])} entries)")


def get_exported_ids(manifest):
    """Get set of already-exported conversation IDs from manifest."""
    return {c["conversation_id"] for c in manifest.get("conversations", [])}


def sanitize_filename(name):
    """Remove or replace characters that are invalid in filenames."""
    # Replace characters not allowed in Windows/Mac/Linux filenames
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    # Collapse multiple underscores/spaces
    name = re.sub(r'[_\s]+', ' ', name).strip()
    # Truncate to reasonable length (leave room for date + .md)
    return name[:80].strip()


def make_filename(title, date):
    """Generate filename: '{date} {title}.md'"""
    safe_title = sanitize_filename(title)
    return f"{date} {safe_title}.md"


def generate_markdown(conversation, messages):
    """Generate Obsidian-compatible Markdown with YAML frontmatter."""
    title = conversation.get("original_title", "Untitled")
    date = conversation.get("date", "unknown-date")
    conv_id = conversation.get("conversation_id", "")
    url = conversation.get("url", "")

    # YAML frontmatter
    # Escape quotes in title for YAML safety
    safe_title = title.replace('"', '\\"')
    lines = [
        "---",
        f'date: "{date}"',
        f'original_title: "{safe_title}"',
        f'conversation_id: "{conv_id}"',
        f'url: "{url}"',
        "---",
        "",
        f"# {title}",
        "",
    ]

    # Conversation body
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        lines.append(f"**{role}**: {content}")
        lines.append("")

    return "\n".join(lines)


def export_conversation(conversation, manifest):
    """Export a single conversation to Markdown. Returns (success, filename_or_error)."""
    conv_id = conversation["conversation_id"]
    title = conversation.get("original_title", "Untitled")
    date = conversation["date"]
    url = conversation["url"]

    try:
        # Fetch messages
        messages = fetch_messages(conv_id, url)
        if not messages:
            return False, f"No messages found for {conv_id[:8]}"

        # Generate markdown
        md_content = generate_markdown(conversation, messages)

        # Write file
        filename = make_filename(title, date)
        filepath = EXPORT_DIR / filename
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(md_content)

        # Update manifest
        manifest["conversations"].append({
            "conversation_id": conv_id,
            "original_title": title,
            "date": date,
            "url": url,
            "file_name": filename,
            "exported_at": datetime.now().isoformat(),
        })

        return True, filename

    except Exception as e:
        log.error(f"  ✗ Failed {conv_id[:8]}: {e}")
        return False, str(e)


print("✓ Markdown generation functions defined")
print("  Filename format: '{date} {title}.md'")
print("  export_conversation(conversation, manifest) → (success, filename_or_error)")

### Export all DeepSeek chat history main workflow
Runs the full export pipeline: skips already-exported conversations (incremental), exports new ones, saves manifest, and logs failures.

In [ ]:
from IPython.display import display, HTML

# ── Main export pipeline ──
manifest = load_manifest()
exported_ids = get_exported_ids(manifest)

# Filter to new conversations only
new_convos = [c for c in conversations if c["conversation_id"] not in exported_ids]
skipped = len(conversations) - len(new_convos)

print(f"Total conversations: {len(conversations)}")
print(f"Already exported:    {skipped}")
print(f"New to export:       {len(new_convos)}")

SHOW_FIRST = 3  # Show first N in plain text, rest collapsed
successes = 0
failures = []
collapsed_lines = []

for i, conv in enumerate(new_convos, 1):
    success, result = export_conversation(conv, manifest)
    status = "✓" if success else "✗"
    line = f"[{i}/{len(new_convos)}] {status} {conv['original_title'][:60]}"

    if success:
        successes += 1
    else:
        failures.append({"conversation_id": conv["conversation_id"],
                         "title": conv.get("original_title", ""),
                         "error": result})
        line += f"  — {result}"

    if i <= SHOW_FIRST:
        print(line)
    else:
        collapsed_lines.append(line)

    # Save manifest after each export (in case of crash)
    save_manifest(manifest)

    # Politeness delay between conversations
    if i < len(new_convos):
        time.sleep(REQUEST_DELAY_SEC)

# Show remaining exports in a collapsible block
if collapsed_lines:
    details_body = "\n".join(collapsed_lines)
    display(HTML(
        f"<details><summary>▸ Show remaining {len(collapsed_lines)} exports...</summary>"
        f"<pre>{details_body}</pre></details>"
    ))

# Write failure log
if failures:
    with open(FAILURE_LOG, "w", encoding="utf-8") as f:
        for fail in failures:
            f.write(f"{fail['conversation_id']} | {fail['title']} | {fail['error']}\n")

print(f"\n{'=' * 50}")
print(f"Export complete!")
print(f"  ✓ Succeeded: {successes}")
print(f"  ✗ Failed:    {len(failures)}")
print(f"  ⊘ Skipped:   {skipped}")
print(f"  Output dir:  {EXPORT_DIR.resolve()}")

### Summary Statistics
Review exported files and manifest contents.

In [ ]:
# ── Summary ──
md_files = list(EXPORT_DIR.glob("*.md"))
manifest_data = load_manifest()

print(f"Markdown files in {EXPORT_DIR.resolve()}:")
print(f"  Total .md files: {len(md_files)}")
print()

# File size stats
if md_files:
    sizes = [f.stat().st_size for f in md_files]
    print(f"  Smallest: {min(sizes):,} bytes")
    print(f"  Largest:  {max(sizes):,} bytes")
    print(f"  Total:    {sum(sizes):,} bytes ({sum(sizes)/1024:.1f} KB)")
    print()

print(f"Manifest entries: {len(manifest_data.get('conversations', []))}")
print(f"Last updated:     {manifest_data.get('last_updated', 'never')}")

# Show failures if any
if FAILURE_LOG.exists():
    with open(FAILURE_LOG, "r", encoding="utf-8") as f:
        fail_lines = f.readlines()
    if fail_lines:
        print(f"\n⚠ {len(fail_lines)} failed exports (see {FAILURE_LOG}):")
        for line in fail_lines[:10]:
            print(f"  {line.strip()}")
else:
    print("\n✓ No failures logged")

### Import to Obsidian
Copies all `.md` files from `DeepSeek_Exports/` into your Obsidian vault subfolder. Only copies files that don't already exist in the vault — safe to re-run.

In [ ]:
import shutil

# ── Validate vault path ──
vault_target = OBSIDIAN_VAULT_DIR / OBSIDIAN_SUBFOLDER
if not OBSIDIAN_VAULT_DIR.exists():
    raise FileNotFoundError(
        f"Obsidian vault not found: {OBSIDIAN_VAULT_DIR}\n"
        "Update OBSIDIAN_VAULT_DIR in the config cell to your actual vault path."
    )

vault_target.mkdir(parents=True, exist_ok=True)

# ── Copy new files only ──
source_files = list(EXPORT_DIR.glob("*.md"))
copied = 0
skipped = 0

for src in source_files:
    dest = vault_target / src.name
    if dest.exists():
        skipped += 1
    else:
        shutil.copy2(src, dest)
        copied += 1

print(f"Obsidian import complete → {vault_target}")
print(f"  ✓ Copied:  {copied} new files")
print(f"  ⊘ Skipped: {skipped} (already exist)")
print(f"  Total in vault subfolder: {sum(1 for f in vault_target.glob('*.md'))}")

------------------------

### Clear Exports
Run ths only if you want to rerun export all from stratch
Delete all files in the `DeepSeek_Exports/` folder (including manifest). Use this to start fresh.

In [ ]:
# import shutil

# if EXPORT_DIR.exists():
#     file_count = sum(1 for f in EXPORT_DIR.iterdir() if f.is_file())
#     shutil.rmtree(EXPORT_DIR)
#     EXPORT_DIR.mkdir(exist_ok=True)
#     print(f"✓ Cleared {file_count} files from {EXPORT_DIR.resolve()}")
# else:
#     print(f"⚠ Export directory doesn't exist: {EXPORT_DIR.resolve()}")